In [1]:
from pathlib import Path
import json
import itertools
import numpy as np
import pandas as pd
import yaml

from ultralytics import YOLO

try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False


In [ ]:
DATA_YAML = Path("C:/Users/felix/Desktop/ELISA/Semestre_9/deeplearning/Projet/Projet_LANDING_PAD_HAFAIEDH_FELIX/data.yaml")                # A changer pour le path de ton data.yaml 
train_images_dir =Path("C:/Users/felix/Desktop/ELISA/Semestre_9/deeplearning/Projet/Projet_LANDING_PAD_HAFAIEDH_FELIX/train/images")       # A changer pour le path de ton dossier train/images 
val_images_dir   = Path("C:/Users/felix/Desktop/ELISA/Semestre_9/deeplearning/Projet/Projet_LANDING_PAD_HAFAIEDH_FELIX/valid/images")      # A changer pour le path de ton dossier valid/images
test_images_dir  =Path("C:/Users/felix/Desktop/ELISA/Semestre_9/deeplearning/Projet/Projet_LANDING_PAD_HAFAIEDH_FELIX/test/images")        # A changer pour le path de ton dossier test/images

print("OK, data.yaml:", DATA_YAML)
print("train:", train_images_dir)
print("val:  ", val_images_dir)
print("test: ", test_images_dir)

OK, data.yaml: data.yaml
train: \train\images
val:   \valid\images
test:  \test\images


In [3]:
BASE_MODEL = "yolov8n.pt"

model = YOLO(BASE_MODEL)

train_results = model.train(
    data=str(DATA_YAML),
    imgsz=640,
    epochs=20,
    batch=16,
    device=0,
    project="runs/detect",
    name="landing_pad",
    exist_ok=True,
)

print("Done. Checkpoint best.pt dans runs/detect/landing_pad/weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.22 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.9  Python-3.12.10 torch-2.8.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=landing_pad, nbs

In [4]:
best_ckpt = Path("runs/detect/runs/detect/landing_pad/weights/best.pt")
assert best_ckpt.exists(), f"best.pt introuvable: {best_ckpt.resolve()}"

model = YOLO(str(best_ckpt))

val_metrics = model.val(data=str(DATA_YAML), split="val", imgsz=640, device=0)
print(val_metrics)

Ultralytics 8.4.9  Python-3.12.10 torch-2.8.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 344.953.3 MB/s, size: 26.8 KB)
val: Scanning C:\Users\felix\Desktop\ELISA\Semestre_9\deeplearning\Projet\Projet_LANDING_PAD_HAFAIEDH_FELIX\valid\labels.cache... 15 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 15/15  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 11, len(boxes) = 12. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.6s/it 2.6s
                   all         15         12          1      0.607      0.868      0.548
Speed: 7.4ms preprocess, 26.2ms inference, 0.0ms loss, 1.3

In [5]:
def list_images(images_dir: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    imgs = [p for p in images_dir.rglob("*") if p.suffix.lower() in exts]
    imgs.sort()
    return imgs

def label_path_from_image(img_path: Path) -> Path:
    # .../valid/images/xxx.jpg -> .../valid/labels/xxx.txt
    parts = list(img_path.parts)
    if "images" not in parts:
        raise ValueError(f"'images' not found in {img_path}")
    idx = parts.index("images")
    parts[idx] = "labels"
    return Path(*parts).with_suffix(".txt")

def has_pad_gt(img_path: Path) -> bool:
    lbl = label_path_from_image(img_path)
    if not lbl.exists():
        return False
    txt = lbl.read_text(encoding="utf-8").strip()
    return len(txt) > 0

def decide_safe(boxes_xyxy: np.ndarray, confs: np.ndarray, w: int, h: int,
                tau_conf: float, tau_area: float):
    """
    Retourne: (safe, max_conf, max_area_ratio)
    """
    if boxes_xyxy.size == 0:
        return False, 0.0, 0.0

    areas = (boxes_xyxy[:, 2] - boxes_xyxy[:, 0]) * (boxes_xyxy[:, 3] - boxes_xyxy[:, 1])
    area_ratio = areas / float(w * h)

    ok = (confs >= tau_conf) & (area_ratio >= tau_area)
    safe = bool(np.any(ok))

    i_best = int(np.argmax(confs))
    return safe, float(confs[i_best]), float(area_ratio[i_best])

In [6]:
from pathlib import Path
import yaml

DATA_YAML = Path("data.yaml")

cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
yaml_dir = DATA_YAML.parent.resolve()

def resolve(p): 
    p = Path(p)
    return p if p.is_absolute() else (yaml_dir / p).resolve()

val_dir = resolve(cfg["val"])
print("val_dir =", val_dir)
print("exists =", val_dir.exists())
print("nb files =", len(list(val_dir.rglob("*"))))
print("nb images jpg/png =", len([p for p in val_dir.rglob("*") if p.suffix.lower() in {".jpg",".jpeg",".png",".webp"}]))

val_dir = C:\Users\felix\Desktop\ELISA\Semestre_9\deeplearning\Projet\valid\images
exists = False
nb files = 0
nb images jpg/png = 0


In [7]:
val_images = list_images(val_images_dir)
print("Nb val images:", len(val_images))

def predict_one(img_path: Path, imgsz=640):
    r = model.predict(source=str(img_path), imgsz=imgsz, conf=0.001, verbose=False)[0]
    im = r.orig_img
    h, w = im.shape[:2]
    if r.boxes is None or len(r.boxes) == 0:
        boxes = np.zeros((0,4), dtype=np.float32)
        confs = np.zeros((0,), dtype=np.float32)
    else:
        boxes = r.boxes.xyxy.cpu().numpy().astype(np.float32)
        confs = r.boxes.conf.cpu().numpy().astype(np.float32)
    return w, h, boxes, confs

cache = []
for p in val_images:
    w, h, boxes, confs = predict_one(p, imgsz=640)
    cache.append((p, w, h, boxes, confs, int(has_pad_gt(p))))

def metrics_binary(y_true, y_pred):
    y_true = np.array(y_true, dtype=int)
    y_pred = np.array(y_pred, dtype=int)
    tp = int(((y_true==1)&(y_pred==1)).sum())
    tn = int(((y_true==0)&(y_pred==0)).sum())
    fp = int(((y_true==0)&(y_pred==1)).sum())
    fn = int(((y_true==1)&(y_pred==0)).sum())
    precision = tp/(tp+fp) if (tp+fp)>0 else 0.0
    recall    = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1        = (2*precision*recall/(precision+recall)) if (precision+recall)>0 else 0.0
    return dict(tp=tp, tn=tn, fp=fp, fn=fn, precision=precision, recall=recall, f1=f1)

tau_conf_grid = np.linspace(0.30, 0.70, 21)
tau_area_grid = np.linspace(0.01, 0.10, 19)

rows = []
for tau_conf, tau_area in itertools.product(tau_conf_grid, tau_area_grid):
    y_true, y_pred = [], []
    for (p, w, h, boxes, confs, gt) in cache:
        safe, _, _ = decide_safe(boxes, confs, w, h, tau_conf=float(tau_conf), tau_area=float(tau_area))
        y_true.append(gt)
        y_pred.append(int(safe))
    m = metrics_binary(y_true, y_pred)
    rows.append({"tau_conf": float(tau_conf), "tau_area": float(tau_area), **m})

df_tune = pd.DataFrame(rows).sort_values(["f1", "recall"], ascending=False).reset_index(drop=True)
df_tune.head(10)

Nb val images: 0


,tau_conf,tau_area,tp,tn,fp,fn,precision,recall,f1
0,0.3,0.010,0,0,0,0,0.0,0.0,0.0
1,0.3,0.015,0,0,0,0,0.0,0.0,0.0
2,0.3,0.020,0,0,0,0,0.0,0.0,0.0
3,0.3,0.025,0,0,0,0,0.0,0.0,0.0
4,0.3,0.030,0,0,0,0,0.0,0.0,0.0
5,0.3,0.035,0,0,0,0,0.0,0.0,0.0
6,0.3,0.040,0,0,0,0,0.0,0.0,0.0
7,0.3,0.045,0,0,0,0,0.0,0.0,0.0
8,0.3,0.050,0,0,0,0,0.0,0.0,0.0
9,0.3,0.055,0,0,0,0,0.0,0.0,0.0


In [8]:
best = df_tune.iloc[0]
tau_conf_best = float(best["tau_conf"])
tau_area_best = float(best["tau_area"])

print("Best thresholds on VAL:")
print("tau_conf =", tau_conf_best)
print("tau_area =", tau_area_best)
print("VAL metrics:", best.to_dict())

Path("runs/postproc").mkdir(parents=True, exist_ok=True)
(Path("runs/postproc/best_thresholds.json")
 .write_text(json.dumps({"tau_conf": tau_conf_best, "tau_area": tau_area_best}, indent=2), encoding="utf-8"))

df_tune.to_csv("runs/postproc/tuning_results.csv", index=False)
print("Saved: runs/postproc/best_thresholds.json and tuning_results.csv")

Best thresholds on VAL:
tau_conf = 0.3
tau_area = 0.01
VAL metrics: {'tau_conf': 0.3, 'tau_area': 0.01, 'tp': 0.0, 'tn': 0.0, 'fp': 0.0, 'fn': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
Saved: runs/postproc/best_thresholds.json and tuning_results.csv


In [9]:
test_images = list_images(test_images_dir)
print("Nb test images:", len(test_images))

y_true, y_pred = [], []
per_img = []

for p in test_images:
    w, h, boxes, confs = predict_one(p, imgsz=640)
    gt = int(has_pad_gt(p))
    safe, max_conf, max_area = decide_safe(boxes, confs, w, h, tau_conf_best, tau_area_best)
    y_true.append(gt)
    y_pred.append(int(safe))
    per_img.append({
        "image": str(p),
        "gt_safe": gt,
        "pred_safe": int(safe),
        "max_conf": max_conf,
        "area_ratio": max_area,
        "n_det": int(boxes.shape[0]),
    })

m_test = metrics_binary(y_true, y_pred)
print("TEST metrics (frozen thresholds):", m_test)

pd.DataFrame(per_img).to_csv("runs/postproc/test_predictions.csv", index=False)
print("Saved: runs/postproc/test_predictions.csv")

Nb test images: 0
TEST metrics (frozen thresholds): {'tp': 0, 'tn': 0, 'fp': 0, 'fn': 0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
Saved: runs/postproc/test_predictions.csv


In [ ]:
if not HAS_CV2:
    print("OpenCV (cv2) non installé -> live désactivé. Installe opencv-python pour l'activer.")
else:
    import time

    tau_conf = tau_conf_best
    tau_area = tau_area_best

    cap = cv2.VideoCapture(0)
    assert cap.isOpened(), "Impossible d'ouvrir la webcam"
    out_dir = Path("runs/postproc")
    out_dir.mkdir(parents=True, exist_ok=True)

    video_path = str(out_dir / "live_demo.mp4")

    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fps = cap.get(cv2.CAP_PROP_FPS)
    fps = fps if fps and fps > 1 else 30.0

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(video_path, fourcc, fps, (W, H))
    assert writer.isOpened(), "Impossible d'ouvrir VideoWriter (essaie AVI/XVID si besoin)"
    print("🎥 Enregistrement vidéo:", video_path)

    logs = []
    t0 = time.time()
    scenario_id = 0 

    print("LIVE: appuie sur 'q' pour quitter, 'n' pour scenario_id+1")

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        r = model.predict(source=frame, imgsz=640, conf=0.001, verbose=False)[0]
        h, w = frame.shape[:2]

        if r.boxes is None or len(r.boxes) == 0:
            boxes = np.zeros((0,4), dtype=np.float32)
            confs = np.zeros((0,), dtype=np.float32)
        else:
            boxes = r.boxes.xyxy.cpu().numpy().astype(np.float32)
            confs = r.boxes.conf.cpu().numpy().astype(np.float32)

        safe, max_conf, max_area = decide_safe(boxes, confs, w, h, tau_conf, tau_area)

        if len(boxes) > 0:
            areas = (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1])
            area_ratio = areas / float(w*h)
            i_best = int(np.argmax(confs))  
            best_conf = float(confs[i_best])
            best_area = float(area_ratio[i_best])
        else:
            i_best = None
            best_conf = 0.0
            best_area = 0.0

        safe = False
        if i_best is not None:
            safe = (best_conf >= tau_conf) and (best_area >= tau_area)

        if i_best is not None:
            x1, y1, x2, y2 = boxes[i_best].astype(int)

            color = (0, 255, 0) if safe else (0, 0, 255) 
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

            cv2.putText(frame, f"conf={best_conf:.2f} area={best_area:.3f}",
                (x1, max(0, y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        
        txt = f"{'SAFE' if safe else 'NOT SAFE'}  conf={max_conf:.2f}  area={max_area:.3f}"
        cv2.putText(frame, txt, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)

        logs.append({
            "timestamp": time.time() - t0,
            "scenario_id": scenario_id,
            "max_conf": max_conf,
            "area_ratio": max_area,
            "tau_conf": tau_conf,
            "tau_area": tau_area,
            "decision": "SAFE" if safe else "NOT_SAFE",
            "notes": ""
        })

        writer.write(frame)

        cv2.imshow("Landing Pad - LIVE", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        if key == ord("n"):
            scenario_id += 1

    cap.release()
    writer.release()
    print("✅ Vidéo sauvegardée :", video_path)
    cv2.destroyAllWindows()

    pd.DataFrame(logs).to_csv("runs/postproc/live_log.csv", index=False)
    print("Saved: runs/postproc/live_log.csv")

🎥 Enregistrement vidéo: runs\postproc\live_demo.mp4
LIVE: appuie sur 'q' pour quitter, 'n' pour scenario_id+1
